# 03 · Tool calling (giving the model hands)

**Course:** Building LLM Agents, notebook 3 of 10
**Where we are:** the pivot of the whole course. So far the model can only produce *text*. Now we let it call **your code**, and a text predictor becomes something that can *act*.

### What you'll learn
- Why a language model **needs tools** at all (it can't do reliable maths, doesn't know today's date, can't read your files)
- How to **define a tool**: a real Python function *plus* its JSON-Schema description in the `tools=[...]` format
- The **three-round trip**: the model decides, then *you* execute, then the model answers
- How to read `finish_reason == "tool_calls"` and parse `message.tool_calls`
- Giving the model **several tools** and letting it pick, including **parallel** tool calls and `tool_choice`
- A first **safety** note: tool arguments come from the model, so you must validate them

### Prerequisites
- You've done **01** (chat completions, messages/roles, `temperature`) and **02** (structured output / JSON, which is how the model will pass us arguments)
- Your `.env` has a working Sigma2 token

> Mental model: the model never runs anything. When it wants a tool, it just emits a structured request, "please call `get_weather` with `{"city": "Oslo"}`". You run the function and hand the result back. The model is the brain that decides; your code is the hands that act. Tool calling is the entire bridge between a chatbot and an agent.

## Setup

Same one-line import as every notebook from here on. `client` is the ready OpenAI client pointed at Sigma2, `MODEL` is the id of your project's model, and `show_messages` renders a conversation so we can *see* the round trip.

In [ ]:
from helpers import client, MODEL, show_messages
import json

## 1. Why the model needs tools

A language model is a next-token predictor (notebook 01). It's astonishing at language, and it genuinely *can't* do some things a five-line Python script does trivially:

- **Maths.** It predicts plausible-looking digits; for anything non-trivial it quietly gets sums wrong.
- **Fresh facts.** It has a training cut-off. It does **not** know today's date, the weather, or your latest order.
- **Your world.** It can't read your files, query your database, or hit your API on its own.

First let's watch it fail at arithmetic, so the fix actually lands.

![Without a tool the model guesses wrong; with a tool it's right](figures/03-why-tools.svg)

*Figure: why tools exist. A confident wrong guess versus a grounded correct result. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
# No tools yet — just ask the model to multiply two biggish numbers.
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is 19 * 4823 * 77? Reply with just the number."}],
    temperature=0,
)
print("model says :", r.choices[0].message.content.strip())
print("python says:", 19 * 4823 * 77)

Maybe it got lucky, maybe not. Either way you can't *trust* it, and "usually right" is useless for software. The fix isn't a better prompt. It's to stop asking the model to *be* a calculator and instead let it *use* one.

> Key idea: a tool is just a normal function you expose to the model. The model decides *when* and *with what arguments* to call it; your code does the actual work and returns a result. The model can't read your files, but it can ask your code to.

## 2. Defining a tool

A tool has **two halves**:

1. **The real Python function.** The code that does the work.
2. **A JSON-Schema description.** What we *tell the model* about that function, so it knows the tool exists, what it does, and what arguments it takes.

Here are two functions we'll give the model. Note that `calculator` already does a little input-checking; we'll come back to *why* in section 7.

In [ ]:
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # A real tool would call a weather API. We fake it so the notebook is offline-safe.
    fake = {
        "Oslo":   "12C, light rain",
        "Bergen": "9C, heavy rain",
        "Tromso": "3C, snow",
    }
    return fake.get(city, f"No weather data for {city!r}")


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '3 * (4 + 5)'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:            # reject anything that isn't plain arithmetic
        return "ERROR: expression contains disallowed characters"
    return str(eval(expression, {"__builtins__": {}}, {}))


# Quick sanity check that our functions work as plain Python:
print(get_weather("Oslo"))
print(calculator("19 * 4823 * 77"))

Now the *description*. This is the part the model sees. The API wants a specific shape, a list of entries, each `{"type": "function", "function": {...}}`, and the inner `function` object has four fields that matter:

| Field | What it is |
|---|---|
| `name` | the function name the model will use to refer to the tool |
| `description` | plain-English explanation of **what the tool does and when to use it** |
| `parameters` | a **JSON Schema** describing the arguments (types, which are `required`) |
| `parameters.properties[*].description` | per-argument hint, *also* read by the model |

> Rule of thumb: these descriptions are prompt engineering (notebook 02), not documentation. The model picks tools and fills arguments purely from this text. Vague descriptions lead to the wrong tool and the wrong args. Write them for the model, not for yourself.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city. Use this for any question about current weather conditions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name, e.g. 'Oslo'."}
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic arithmetic expression. Use this for ANY arithmetic instead of computing it yourself.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "An arithmetic expression, e.g. '3 * (4 + 5)'."}
                },
                "required": ["expression"],
            },
        },
    },
]

# A small lookup so we can find the real function by the name the model uses.
TOOL_FUNCS = {"get_weather": get_weather, "calculator": calculator}

![The four-message tool-calling round trip](figures/03-round-trip.svg)

*Figure: the four-message round trip behind a single tool call. The model only requests; your code executes. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 3. Round 1, the model decides

We make a normal `chat.completions.create(...)` call, but pass `tools=tools`. Everything else is as in notebook 01. We keep `temperature=0` because choosing a tool is a *decision*, and decisions should be predictable.

We'll start with a question that needs **both** tools, to see what happens.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use the available tools when they apply."},
    {"role": "user",   "content": "What's the weather in Oslo right now, and what is 19 * 4823 * 77?"},
]

r1 = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,        # <-- the only new ingredient
    temperature=0,
)

msg = r1.choices[0].message
print("finish_reason:", r1.choices[0].finish_reason)
print("content      :", repr(msg.content))   # often None when the model wants a tool

Two things to notice:

- **`finish_reason == "tool_calls"`.** Back in notebook 01 we saw `"stop"` (finished) and `"length"` (hit the cap). This is the third one: *"I don't want to answer yet, I want to call a tool."*
- **`content` is usually `None`.** Instead of text, the model produced a **structured request**. That request lives in `msg.tool_calls`.

Let's open it up.

In [ ]:
for call in msg.tool_calls:
    print("tool_call id :", call.id)
    print("  type       :", call.type)
    print("  name       :", call.function.name)
    print("  arguments  :", repr(call.function.arguments), "  <-- a JSON *string*, not a dict")
    print()

The model decided to call **both** tools at once. That's a **parallel tool call**, and our endpoint supports it. (More on that in section 6.)

> Gotcha: `call.function.arguments` is a JSON string, not a Python dict. That's deliberate; it's the structured output from notebook 02, arriving as text. You must `json.loads(...)` it before you can use it. In production you'd wrap that in a `try/except`, since the model *usually* emits valid JSON, but "usually" isn't "always".

In [ ]:
args = json.loads(msg.tool_calls[0].function.arguments)
print("parsed:", args, " | type:", type(args).__name__)

> Why you care: the model has not run anything. It can't; it has no Python interpreter. It only *asked*. Nothing happens to your files, your database, or your bank account unless **you** choose to act on the request. That control point is the whole safety story of agents, and it's why you, not the model, are holding the steering wheel.

## 4. Round 2, you execute and report back

Now it's our turn. For each requested call we:

1. parse the JSON arguments,
2. look up the real Python function and run it,
3. record the result as a new message with **`role: "tool"`**, tagged with the **same `tool_call_id`** so the model knows which request it answers.

But first we must append the **assistant's own turn**, the message containing the tool-call requests, to the history. The conversation has to read as: *assistant asked for tools, then here are the tool results.* Skip the assistant turn and the next call breaks.

> The verified append pattern (memorise this):
> ```python
> messages.append(msg.model_dump(exclude_none=True))                       # the assistant's request
> messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})  # each result
> ```
> `msg.model_dump(exclude_none=True)` turns the SDK message object back into a plain dict (dropping empty fields like the `None` content) so it can go straight back into the messages list.

In [ ]:
# 1) Append the assistant turn that contains the tool-call requests.
messages.append(msg.model_dump(exclude_none=True))

# 2) Execute each requested call and append a tool-role result.
for call in msg.tool_calls:
    name = call.function.name
    args = json.loads(call.function.arguments or "{}")
    result = TOOL_FUNCS[name](**args)          # <-- YOUR code actually runs here
    print(f"executed {name}({args}) -> {result!r}")
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,                # ties the result to the request
        "content": str(result),                 # tool content must be a string
    })

Our message list now holds four turns: **system, then user, then assistant (tool requests), then tool results**. The model has everything it needs to write a grounded answer; it just hasn't seen the results yet. That's round 3.

## 5. Round 3, the final answer

We call the model **again** with the *same growing `messages` list*, now including the tool results. This time it has the facts it was missing, so it stops asking for tools and writes a natural-language answer.

In [ ]:
r2 = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    temperature=0,
)
final = r2.choices[0].message

print("finish_reason:", r2.choices[0].finish_reason)   # back to "stop" — it's done
print("answer       :", final.content)

# Append the final answer so the history is complete.
messages.append(final.model_dump(exclude_none=True))

`finish_reason` is back to **`"stop"`**, the model is satisfied and needs no more tools, and the answer is **grounded in the tool output**: the real weather we returned, and the *exactly correct* product from our calculator, not a hallucinated number.

Let's see the entire round trip in one view. This is the picture to keep in your head for the rest of the course.

In [ ]:
show_messages(messages)

Read it top to bottom; that is the complete loop:

1. The **user** asks a question.
2. The **assistant** doesn't answer; it *requests two tools*.
3. The **tool** messages carry the results **we** computed.
4. The **assistant** writes the final, grounded answer.

> Key idea: tool calling is a conversation, not a function return. The tool results go back in as *messages*, and the model reads them like any other turn. That's why everything still flows through `messages`, the one list you've been maintaining since notebook 01.

## 6. Several tools, and letting the model choose

We already handed the model two tools and it picked the right one (or ones) for each part of the question. That's the real power: give it a **toolbox** and a good system prompt, and it routes each request to the appropriate tool.

A weather-only question should trigger only `get_weather`:

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Is it raining in Bergen?"}],
    tools=tools,
    temperature=0,
)
chosen = [c.function.name for c in (r.choices[0].message.tool_calls or [])]
print("finish_reason:", r.choices[0].finish_reason)
print("tools chosen :", chosen)

And a maths-only question triggers only `calculator`. The model reads our `description` fields and routes accordingly, which is exactly why those descriptions are worth getting right.

Two more controls worth knowing:

- **Parallel tool calls.** As we saw in section 3, a single response can contain **several** `tool_calls`. Always loop over `msg.tool_calls` (even if you expect one) and append one `tool` message per call. The model can fan out, and you handle them all before round 3.
- **`tool_choice`** lets *you* override the model's discretion:
  - `"auto"` (the default): the model decides whether to use a tool.
  - `"none"`: tools are visible but **forbidden**; the model must answer in text.
  - `{"type": "function", "function": {"name": "calculator"}}`: **force** a specific tool.

In [ ]:
# tool_choice="none": the model may NOT call a tool, even though it can see them.
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is 2 + 2? Answer in one word."}],
    tools=tools,
    tool_choice="none",
    temperature=0,
    max_tokens=8000,
)
print("finish_reason:", r.choices[0].finish_reason)   # "stop" — no tool call was allowed
print("answer       :", repr(r.choices[0].message.content))

## 7. A safety note: tool arguments come from the model

Look again at what happens in round 2: we take `call.function.arguments`, text the **model** produced, and feed it straight into our function. The model is usually sensible, but it is **not a trusted caller**. Treat every argument the way you'd treat input from a web form: validate it.

This is why our `calculator` checks its input *before* doing anything:

```python
allowed = set("0123456789+-*/(). ")
if not set(expression) <= allowed:
    return "ERROR: expression contains disallowed characters"
```

Without that guard, `eval` on model-supplied text is a remote-code-execution hole waiting to happen. (Even with the character filter, raw `eval` is only acceptable for a teaching demo; a real calculator tool would use a proper expression parser.)

> Gotcha: never pass model output into `eval`, `exec`, `os.system`, raw SQL, or a file path without validating it first. The model can hallucinate arguments, and a malicious *user* can talk the model into emitting dangerous ones. You'll harden tools properly, with input validation, allow-lists, and returning errors instead of crashing, in **notebook 06**.

## 8. Look at what we just did, that's an agent (almost)

We ran the round trip **by hand**: one model call, parse the tool requests, execute, feed results back, one more model call. But notice the *shape* of it:

> **call the model, did it ask for tools? If yes, run them and loop back; if no, we're done.**

We did that loop exactly **once**. What if the model needs a *second* round of tools, for example it gets the weather, then decides it needs a calculation based on it? Right now we'd have to copy-paste the round-2 and round-3 code again.

> Mental model: an agent is precisely this manual round trip, wrapped in a `while` loop with a step limit. Keep calling the model and running its tools until it stops asking. Nothing more magical than that.

That's notebook 04. We'll take the exact code from this notebook and turn it into a small reusable loop, and at that moment you'll have built your first agent.

## Recap

You can now:
- explain **why** a model needs tools (maths, fresh facts, access to your world)
- **define a tool**: a Python function **plus** its `{"type":"function","function":{...}}` JSON-Schema description
- run **round 1**: pass `tools=`, detect `finish_reason == "tool_calls"`, and parse `msg.tool_calls` (arguments are a **JSON string**, so use `json.loads`)
- run **round 2**: append the assistant turn with `msg.model_dump(exclude_none=True)`, execute, and append each result as `{"role":"tool","tool_call_id":...,"content":...}`
- run **round 3**: call again to get the final, **grounded** answer (`finish_reason == "stop"`)
- give the model **multiple tools**, handle **parallel** calls, and steer with **`tool_choice`**
- remember that tool arguments come from the model, so validate them

### Try it yourself
1. Add a third tool, `get_time(timezone)`, that returns a fixed string, and ask a question that needs all three. Confirm the model calls each one.
2. **Force** a tool with `tool_choice={"type": "function", "function": {"name": "calculator"}}` on a *weather* question. What does the model put in `expression`? (This is why forcing tools is risky.)
3. Remove the character-filter guard from `calculator` and pass `expression="__import__('os').getcwd()"`. See why validation matters, then put the guard back.
4. Make `get_weather` raise an exception, run the round trip, and append the error string as the tool `content`. Watch the model react to a failed tool. (That's the error-handling we formalise in nb 06.)

### Next up
**04 · Your first agent.** We take this exact manual round trip and wrap it in a loop with a step limit. An agent is *this*, in a `while`-loop. By the end you'll have a working `Agent` that keeps calling tools until the task is done.